In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("PortDataCleaning").master("local[*]").getOrCreate()

df = spark.read.option("header", True).option("inferSchema", True).csv("../data/raw/lk-port-activity.csv")
print("Original records: ", df.count())

d:\Big data project\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Original records:  11412


In [2]:
df_clean = df.withColumn("date", to_date(col("date")))
#df_clean.printSchema()
df_clean.show(10)

+----------+----+-----+---+--------+-----------+---------+----+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+-----------+-------------+------------+------+----------------+---------------+--------------------+-----------+-------------+------------+------+
|      date|year|month|day|  portid|   portname|  country|ISO3|portcalls_container|portcalls_dry_bulk|portcalls_general_cargo|portcalls_roro|portcalls_tanker|portcalls_cargo|portcalls|import_container|import_dry_bulk|import_general_cargo|import_roro|import_tanker|import_cargo|import|export_container|export_dry_bulk|export_general_cargo|export_roro|export_tanker|export_cargo|export|
+----------+----+-----+---+--------+-----------+---------+----+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+-------------

In [3]:
missing = df_clean.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c) for c in df_clean.columns
])
missing.show()

+----+----+-----+---+------+--------+-------+----+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+-----------+-------------+------------+------+----------------+---------------+--------------------+-----------+-------------+------------+------+
|date|year|month|day|portid|portname|country|ISO3|portcalls_container|portcalls_dry_bulk|portcalls_general_cargo|portcalls_roro|portcalls_tanker|portcalls_cargo|portcalls|import_container|import_dry_bulk|import_general_cargo|import_roro|import_tanker|import_cargo|import|export_container|export_dry_bulk|export_general_cargo|export_roro|export_tanker|export_cargo|export|
+----+----+-----+---+------+--------+-------+----+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+-----------+-------------+-----

In [4]:
numeric_columns = [
    c for c,t in df_clean.dtypes
    if t in ["int", "bigint", "double", "float", "long"]
]

for c in numeric_columns:
    negative_count = df_clean.filter(
        col(c)<0
    ).count()
    if negative_count>0:
        print(c, " has ", negative_count, " negative values")

In [5]:
df_clean = df.drop("country", "ISO3")
df_clean.show()

+-------------------+----+-----+---+--------+-----------+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+-----------+-------------+------------+------+----------------+---------------+--------------------+-----------+-------------+------------+------+
|               date|year|month|day|  portid|   portname|portcalls_container|portcalls_dry_bulk|portcalls_general_cargo|portcalls_roro|portcalls_tanker|portcalls_cargo|portcalls|import_container|import_dry_bulk|import_general_cargo|import_roro|import_tanker|import_cargo|import|export_container|export_dry_bulk|export_general_cargo|export_roro|export_tanker|export_cargo|export|
+-------------------+----+-----+---+--------+-----------+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+----------

In [ ]:
df_clean.write.mode("overwrite").parquet("../data/processed/port_activity_clean.parquet")
print("Cleaned dataset saved successfully!")

Py4JJavaError: An error occurred while calling o325.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

In [11]:
print(
    spark.sparkContext._jvm
    .org.apache.hadoop.util.VersionInfo.getVersion()
)

3.5.0
